# KWS eval part 1: Label posteriors with ZIPA
In this notebook I've given you several pieces of the puzzle for how to get KWS probability for Tira audio using the ZIPA model.
I've broken it down into 3 stages:
1. Loading KWS sentences and audio
2. Computing logits with ZIPA
3. Getting label posteriors for each sentence using CTC loss

We don't quite get to actual keyword hit probability in this exercise, but we'll get there next.

## Objective
As you go through this exercise, rather than just fill out the notebook, I'm asking you to create a python script (I've left a blank file in `scripts/eval/kws_ctc_eval.py`) that will get keyword probabilities for every sentence in the dataset.
Think about the components I've given you here, and how they can be arranged in a logical sequence so that you can write a single script that when run end-to-end will perform all the steps needed to evaluate the ZIPA model on our Tira KWS dataset.

## Next steps
After you finish this task the next objective will be to swap from naive label probability to keyword hit probability, and then to use the probabilities to compute evaluation metrics like mAP, F1 and AUROC.

In [22]:
from tira_kws.dataloading import load_capstone_kws_cuts, get_k2_dataloader
from tira_kws.models.zipa import load_zipa_small_crctc
from tira_kws.constants import CAPSTONE_SUPERVISIONS, CAPSTONE_KEYWORDS
from lhotse import SupervisionSet
import pandas as pd
import torch

## Prior reading
Please read this [HuggingFace chapter](https://huggingface.co/learn/llm-course/en/chapter3/4) and [PyTorch tutorial](https://sebastianraschka.com/teaching/pytorch-1h/_) before trying to understand any of the code here, and use them as references if anything here is confusing.

## Keywords
Keywords are stored in a CSV file.
We can load this with Pandas.

In [23]:
df = pd.read_csv(CAPSTONE_KEYWORDS)
df

,keyword,keyword_id,gloss
0,àpɾí,0.0,boy
1,ðɔ̀mɔ̀cɔ̀,1.0,man
2,lɛ́ŋgɛ́n,2.0,mother
3,ŋɔ̀ɽíŋgɔ́,3.0,donkey
4,mùðù,4.0,leopard
5,ùɾnɔ̀,5.0,grandfather/grandchild
6,ðàŋàl,6.0,sheep
7,kə̀və̀lɛ̀ðɔ́,7.0,CLg-pull.PFV-VENT
8,kàŋú,8.0,CLg-see.PFV-VENT
9,íŋgá_nɔ̀nà,9.0,1sg-CLg-AUX_see.IPFV-IT


In [24]:
keywords = df['keyword']
type(keywords)
print(keywords[:5])


0        àpɾí
1     ðɔ̀mɔ̀cɔ̀
2      lɛ́ŋgɛ́n
3    ŋɔ̀ɽíŋgɔ́
4        mùðù
Name: keyword, dtype: object


We'll save the list of keyword strings for easy reference later.

In [25]:
keywords = df['keyword'].tolist()
keywords

['àpɾí',
 'ðɔ̀mɔ̀cɔ̀',
 'lɛ́ŋgɛ́n',
 'ŋɔ̀ɽíŋgɔ́',
 'mùðù',
 'ùɾnɔ̀',
 'ðàŋàl',
 'kə̀və̀lɛ̀ðɔ́',
 'kàŋú',
 'íŋgá_nɔ̀nà']

## Sentence data
KWS sentence metadata are stored in JSONL (= Javascript Object Notation Lines), where every line is a single JSON object which is written similarly to a Python dictionary and can be read as a Python dictionary directly. Let's print the first two lines of the JSONL file using the `json` library for formatting.

In [26]:
import json
with open(CAPSTONE_SUPERVISIONS) as f:
    lines = f.readlines()[:1]
json_objects = [json.loads(line) for line in lines]
json_objects_formatted = [
    json.dumps(obj, indent=2, ensure_ascii=False)
    for obj in json_objects
]
jsonl_header = "\n".join(json_objects_formatted)
print(jsonl_header)

{
  "id": "àpɾí_484_positive",
  "start": 1969.12,
  "duration": 2.851,
  "channel": 0,
  "supervisions": [
    {
      "id": "àpɾí_484_positive",
      "recording_id": "HH02122021",
      "start": 0.0,
      "duration": 2.851,
      "channel": 0,
      "text": "àpɾí jǎŋîvlɛ̀ðɔ́ ǹdòbàgɛ̀",
      "language": "tic",
      "speaker": "Himidan",
      "custom": {
        "fst_text": "àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀",
        "gloss": "boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]",
        "root": "àpɾí ŋgá vəlɛð nd̪ɔ̀bàgɛ̀",
        "translation": "The boy will pull me here tomorrow.",
        "keyword": "àpɾí",
        "record_type": "positive"
      }
    }
  ],
  "recording": {
    "id": "HH02122021",
    "sources": [
      {
        "type": "file",
        "channels": [
          0
        ],
        "source": "/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH02122021.wav"
      }
    ],
    "sampling_rate": 16000,
    "num_samples": 5565

Rather than loading JSONL metadata manually, we use the Lhotse library which is designed for managing speech datasets.
The function `tira_kws/dataloading.py#load_capstone_kws_cuts()` uses Lhotse to load KWS data into a `CutSet` object (i.e. a set of audio snippets cut out from the source audio).
As we can see, the cuts contain the same data as the JSONL file above.

Read the [Lhotse documentation](https://lhotse.readthedocs.io/en/latest/cuts.html) for more information on `Cut` and `CutSet` objects.
By default Lhotse returns a generator rather than a list.
To load cuts as a list instead (so we can index them), call `cuts.to_eager()`.

In [27]:
cuts = load_capstone_kws_cuts()
cuts = cuts.trim_to_supervisions()
cuts = cuts.to_eager()
cuts[0]

MonoCut(id='àpɾí_484_positive', start=1969.12, duration=2.851, channel=0, supervisions=[SupervisionSegment(id='àpɾí_484_positive', recording_id='HH02122021', start=0.0, duration=2.851, channel=0, text='àpɾí jǎŋîvlɛ̀ðɔ́ ǹdòbàgɛ̀', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]', 'root': 'àpɾí ŋgá vəlɛð nd̪ɔ̀bàgɛ̀', 'translation': 'The boy will pull me here tomorrow.', 'keyword': 'àpɾí', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH02122021', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH02122021.wav')], sampling_rate=16000, num_samples=55650022, duration=3478.126375, channel_ids=[0], transforms=None), custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]', 'r

In [28]:
cuts = load_capstone_kws_cuts()
print(cuts)
cuts[0]
first_cut = cuts[0]
first_cut

CutSet(len=203) [underlying data type: <class 'list'>]


MonoCut(id='àpɾí_484_positive', start=1969.12, duration=2.851, channel=0, supervisions=[SupervisionSegment(id='àpɾí_484_positive', recording_id='HH02122021', start=0.0, duration=2.851, channel=0, text='àpɾí jǎŋîvlɛ̀ðɔ́ ǹdòbàgɛ̀', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]', 'root': 'àpɾí ŋgá vəlɛð nd̪ɔ̀bàgɛ̀', 'translation': 'The boy will pull me here tomorrow.', 'keyword': 'àpɾí', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH02122021', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH02122021.wav')], sampling_rate=16000, num_samples=55650022, duration=3478.126375, channel_ids=[0], transforms=None), custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]', 'r

In [29]:
    # load keywords and make them into a list; show first 5 keywords
    df = pd.read_csv(CAPSTONE_KEYWORDS)
    keywords = list(df['keyword'])
    print(keywords[:5])

    # this function loads CAPSTONE_SUPERVISIONS and divides it into cuts
    # one cut = one sentence
    cuts = load_capstone_kws_cuts()
    print(cuts)

    # accessing first cut
    first_cut=cuts[0] 
    print('FIRST CUT')
    print(first_cut)
    # accessing the keyword from the first cut
    keyword_from_first_cut = first_cut.custom['keyword'] 
    print('KEYWORD FROM FIRST CUT')
    print(keyword_from_first_cut)
    # accessing the record type (positive, close-negavtie, negative) 
    record_type_from_first_cut = first_cut.custom['record_type'] 
    print(record_type_from_first_cut)

    # creating the dataloader and passing the cuts
    dataloader = get_k2_dataloader(cuts)

    print('\nNOW WORKING WITH BATCHES')
    # batching
    # batch - supervision dict - individual cut
    for batch in dataloader:
        supervisions_first_batch = batch['supervisions']
        # list of cuts in the first batch
        print(supervisions_first_batch['cut'])
        # accessing one cut in a batch
        first_cut_in_batch = supervisions_first_batch['cut'][0]
        print(first_cut_in_batch)
        # printing the keyword from firt cut in a batch
        print('KEYWORD & RECORD TYPE from first cut first batch')
        print(first_cut_in_batch.custom['keyword'])
        # accessing the record type for that first cut
        print(first_cut_in_batch.custom['record_type'])

        # model input
        # accessing the inputs in a batch
        print('\nMODEL INPUT')
        print(type(batch['inputs']))
        # 1D: batch size, 2D: sequence length/number of timesteps per sentence, 3D: acoustic features per frame
        print(batch['inputs'].shape)
        break

['àpɾí', 'ðɔ̀mɔ̀cɔ̀', 'lɛ́ŋgɛ́n', 'ŋɔ̀ɽíŋgɔ́', 'mùðù']
CutSet(len=203) [underlying data type: <class 'list'>]
FIRST CUT
MonoCut(id='àpɾí_484_positive', start=1969.12, duration=2.851, channel=0, supervisions=[SupervisionSegment(id='àpɾí_484_positive', recording_id='HH02122021', start=0.0, duration=2.851, channel=0, text='àpɾí jǎŋîvlɛ̀ðɔ́ ǹdòbàgɛ̀', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]', 'root': 'àpɾí ŋgá vəlɛð nd̪ɔ̀bàgɛ̀', 'translation': 'The boy will pull me here tomorrow.', 'keyword': 'àpɾí', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH02122021', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH02122021.wav')], sampling_rate=16000, num_samples=55650022, duration=3478.126375, channel_ids=[0], transforms=None), custom=

## Data filtering
It's useful to load all sentences for a particular keyword rather than all sentences at once.

In [30]:
keyword = keywords[1]
keyword_cuts = cuts.filter(
    lambda cut: cut.custom.get('keyword', None) == keyword
)
keyword_cuts = keyword_cuts.to_eager()
keyword, keyword_cuts

('ðɔ̀mɔ̀cɔ̀', CutSet(len=20) [underlying data type: <class 'list'>])

We can also load only the negative records from the dataset.

In [31]:
negative_cuts = cuts.filter(
    lambda cut: cut.custom.get('record_type', None) == 'negative'
)
negative_cuts = negative_cuts.to_eager()
negative_cuts

CutSet(len=3) [underlying data type: <class 'list'>]

## Dataloading and batching
We use a dataloader to serve batches to the model.

In [32]:
dataloader = get_k2_dataloader(cuts)
dataloader

In [33]:
for batch in dataloader:
    print(batch.keys())
    supervisions_first_batch = batch['supervisions']
    print(supervisions_first_batch)
    print(type(supervisions_first_batch))
    print(supervisions_first_batch.keys())
    print(supervisions_first_batch['cut'])
    break

dict_keys(['inputs', 'supervisions'])
{'text': ['ɛ́dɛ́rɛ́n ɛ́gɛ́n kìcə̀lò', 'íŋgánɔ̀nà lɛ́ɲɛ̀', 'íŋgánɔ́nà óɾɛ́ nd̪ɔ̀bà', 'ðàŋàl ðìcə̀lò', 'mùðù kəcó ŋámɽáɾè', 'lɛ́ŋgɛ́n kɔ̀là', 'ŋɔ̀ɽíŋgɔ́ ŋàmɔ̀rðápríɲâ', 'íŋgánɔ̀nà lèdéɲâ', 'kúkù kǎjjì àpɾíɲá ŋɔ̀mɔ̀', 'íŋgánɔ̀nà nɛ́ɾá nd̪ɔ̀bà', 'àn kúkù vɽàn káŋú t̪ólìɲǎ', 'íŋgánɔ̀nà nɛ̀lɛ̀ nd̪ɔ̀bà', 'lɛ́lɛ́n lìcə̀lò', 'kúkù kàŋú t̪ólèɲà jû', 'rɔ̀mɔ̀cɔ̀ rə̀bùlú', 'án ɔ́ɟɔ́ kə́və́lɛ̀ðɔ̀', 'kàŋɔ́ ìlɛ́ðɛ́', 'lɛ́ŋgáí kìcə̀lò', 'ábŕtɔ́ sàɽíŋgɔ̀', 'mùðù kèclò', 'kàðɛ́ŋɛ́n lìcə̀lò', 'ùrnɔ̀ kɔ̀rmùðɔ̀', 'jɛ́ûrmùðɔ̀', 'ðàŋàl ðɔ̂n ðə̀ɽà', 'lùrnɔ̀ lə̀vràðɔ́ ðúdò', 'ðɛ́ŋɛ́n ðìcə̀lò', 'ùrnɔ̀ kìcə̀lò', 'ŋùðì ŋɔ̀là', 'ǎn ðɔ̀mɔ̀cɔ̀ ðɔ̀n', 'sɔ̀ɽíŋgɔ́ sátàɽán', 'ɔ̂rnɔ̀ ɔ́ɟɔ̀', 'jɔ̂rnɔ́ ímìdân'], 'sequence_idx': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31],
    

In [34]:
    for batch in dataloader:
        supervisions_first_batch = batch['supervisions']
        # list of cuts in the first batch
        print(supervisions_first_batch['cut'])
        # one cut in a batch
        print(supervisions_first_batch['cut'][0])
        break

[MonoCut(id='lɛ́ŋgɛ́n_11908_close_negative', start=223.627, duration=1.7, channel=0, supervisions=[SupervisionSegment(id='lɛ́ŋgɛ́n_11908_close_negative', recording_id='HH20230407', start=0.0, duration=1.7, channel=0, text='ɛ́dɛ́rɛ́n ɛ́gɛ́n kìcə̀lò', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'íd̪ɛ́rɛ́n ɛ́gɛ́n kìcə̀lò', 'gloss': 'maternal.aunt/uncle[NOM,SG,3SG/3PL] 3pl.poss[CLG] good[CLG]', 'root': 'd̪ɛ́r ɛ́<CL>ɛ́n icəlo', 'translation': 'their uncle is good', 'keyword': 'lɛ́ŋgɛ́n', 'record_type': 'close_negative'}, alignment=None)], features=None, recording=Recording(id='HH20230407', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH20230407.wav')], sampling_rate=16000, num_samples=28416173, duration=1776.0108125, channel_ids=[0], transforms=None), custom={'fst_text': 'íd̪ɛ́rɛ́n ɛ́gɛ́n kìcə̀lò', 'gloss': 'maternal.aunt/uncle[NOM,SG,3SG/3PL] 3pl.poss[CLG] good[CLG]', 'root': 'd̪ɛ́r ɛ́<CL

The dataloader batches the data into a roughly fixed size (no more than 32 records by default).
Each batch has 'inputs' (the audio features to be fed to the model, stored as torch tensors) and 'supervisions' (the metadata for each record in the batch).

See [Torch documentation](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) for more on dataloaders.

In [35]:
for i, batch in enumerate(dataloader):
    if i >= 5:
        break
    print(batch.keys())
    # input shape is batch_size, sequence_length, feature_dimension
    print(batch['inputs'].shape)

dict_keys(['inputs', 'supervisions'])
torch.Size([32, 170, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([27, 203, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([8, 159, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([25, 227, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([6, 727, 80])


Let's look more closely at the metadata stored under "supervisions."

In [36]:
first_batch = next(iter(dataloader))
supervisions = first_batch['supervisions']
supervisions.keys()

dict_keys(['text', 'sequence_idx', 'start_frame', 'num_frames', 'cut'])

In [37]:
        first_batch = next(iter(dataloader))
        first_cut = first_batch["supervisions"]["cut"][0]
        keyword_1cut_1batch = first_cut.custom['keyword']
        print(keyword_1cut_1batch)

lɛ́ŋgɛ́n


In [59]:
first_cut_input = first_batch['inputs'][0:1]
first_cut_input

tensor([[[-15.1436, -14.3586, -13.3797,  ..., -11.4122, -12.0600, -11.8911],
         [-15.9424, -15.9424, -15.1702,  ..., -11.3341, -12.1784, -12.1412],
         [-15.9424, -15.9424, -15.9139,  ..., -11.5461, -12.1401, -12.3786],
         ...,
         [-13.2275, -13.4283,  -8.7543,  ..., -11.9060, -11.2188, -10.8770],
         [-14.2187, -13.4486,  -9.7652,  ..., -11.7420, -11.1828, -11.8004],
         [-15.9424, -15.2219, -10.6141,  ..., -10.7911, -10.0118, -11.1013]]])

In [60]:
first_cut_input_length = first_batch['supervisions']['num_frames'][0:1]
first_cut_input_length

tensor([170], dtype=torch.int32)

In [62]:
first_cut_embed, first_cut_output_length = model.forward_encoder(first_cut_input, first_cut_input_length)

RuntimeError: Expected all tensors to be on the same device, but got weight is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA___slow_conv2d_forward)

In [ ]:
ctc_logits_for_first_cut = model.ctc_output(first_cut_embed)

first_cut_log_probs=torch.nn.functional.log_softmax(ctc_logits_for_first_cut, dim=2)

NameError: name 'first_cut_embed' is not defined

All records in a batch are automatically padded to the same length, but they were not the same length originally.
Lhotse includes a 'num_frames' field that lets us know the original length of each record.

In [38]:
supervisions['num_frames']

tensor([170, 170, 169, 168, 168, 167, 167, 166, 164, 163, 162, 160, 160, 156,
        155, 153, 151, 143, 142, 141, 139, 138, 137, 132, 129, 125, 121, 114,
        108, 105, 101,  97], dtype=torch.int32)

`batch['supervisions']['cut']` is a list of `lhotse.MonoCut` objects, one for each record in the batch.
We can access the record's metadata (transcription, keyword, whether it's positive or negative) from the `.custom` field of the `MonoCut` object.

In [39]:
print(len(supervisions['cut']))
print(supervisions['cut'][0])

32
MonoCut(id='lɛ́ŋgɛ́n_11908_close_negative', start=223.627, duration=1.7, channel=0, supervisions=[SupervisionSegment(id='lɛ́ŋgɛ́n_11908_close_negative', recording_id='HH20230407', start=0.0, duration=1.7, channel=0, text='ɛ́dɛ́rɛ́n ɛ́gɛ́n kìcə̀lò', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'íd̪ɛ́rɛ́n ɛ́gɛ́n kìcə̀lò', 'gloss': 'maternal.aunt/uncle[NOM,SG,3SG/3PL] 3pl.poss[CLG] good[CLG]', 'root': 'd̪ɛ́r ɛ́<CL>ɛ́n icəlo', 'translation': 'their uncle is good', 'keyword': 'lɛ́ŋgɛ́n', 'record_type': 'close_negative'}, alignment=None)], features=None, recording=Recording(id='HH20230407', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH20230407.wav')], sampling_rate=16000, num_samples=28416173, duration=1776.0108125, channel_ids=[0], transforms=None), custom={'fst_text': 'íd̪ɛ́rɛ́n ɛ́gɛ́n kìcə̀lò', 'gloss': 'maternal.aunt/uncle[NOM,SG,3SG/3PL] 3pl.poss[CLG] good[CLG]', 'root': 'd̪ɛ́r ɛ́<

In [40]:
print(supervisions['cut'][0].custom)

{'fst_text': 'íd̪ɛ́rɛ́n ɛ́gɛ́n kìcə̀lò', 'gloss': 'maternal.aunt/uncle[NOM,SG,3SG/3PL] 3pl.poss[CLG] good[CLG]', 'root': 'd̪ɛ́r ɛ́<CL>ɛ́n icəlo', 'translation': 'their uncle is good', 'keyword': 'lɛ́ŋgɛ́n', 'record_type': 'close_negative', 'dataloading_info': {'rank': 0, 'world_size': 1, 'worker_id': None}}


## Model
I've sequestered some of the code from ZIPA in `tira_kws/models/zipa.py` which we can use to load some of the ZIPA models.
For now let's stick with Zipa-Cr-Small.
Don't forget to put the model to GPU after loading, otherwise inference will be extremely slow.

In [41]:
model = load_zipa_small_crctc()
model = model.to('cuda')
type(model)

model.AsrModel

Here's a sample code snippet for getting CTC logits for a batch from the dataloader.

In [42]:
first_batch = next(iter(dataloader))

with torch.no_grad(): # what does 'no_grad' mean and why are we using it here?
    inputs = first_batch['inputs'].to('cuda')
    input_lengths = first_batch['supervisions']['num_frames'].to('cuda')
    # why put inputs and input_lens to cuda?
    # Hint: try running the snippet without `.to('cuda')`

    # Embeddings = hidden representation prior to output layer
    embeds, output_lengths = model.forward_encoder(inputs, input_lengths)
    # CTC output = logits from final layer
    ctc_logits = model.ctc_output(embeds)
    

## Tokenization
Before we can get keyword probabilities, we need to encode our keywords as token ids.
We can do this using the ZIPA sentencepiece tokenizer.

In [43]:
import sentencepiece
from tira_kws.constants import ZIPA_SENTENCEPIECE_MODEL

tokenizer = sentencepiece.SentencePieceProcessor()
tokenizer.load(str(ZIPA_SENTENCEPIECE_MODEL))

True

In [44]:
tokenizer.encode(['àpɾí', 'jícə̀lò'])

[[3, 4, 2, 19, 80, 12, 2], [3, 13, 12, 2, 6, 50, 2, 15, 18, 2]]

In [45]:
# TODO: encode every keyword from the first batch
# YOUR CODE HERE
encoded_batch_labels = ...

## CTC loss
CTC loss is equivalent to the probability of a label given the logits.
For KWS inference we want to get the probability of a label occurring *within* audio (along other speech), so this isn't quite what we're looking for but it's a good first step.
We'll this approach working before we think about how to get the KWS probability.

To get CTC loss, we need the label posteriors, targets(=labels), input lengths and target lengths.
To convert logits to probabilities, use softmax.(https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html#torch.nn.functional.log_softmax).


See the [PyTorch documentation on CTC loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.ctc_loss.html#torch.nn.functional.ctc_loss) for more info.

In [46]:
output_lengths.shape, ctc_logits.shape

(torch.Size([32]), torch.Size([32, 81, 127]))

In [47]:
from torch.nn.functional import log_softmax
from torch.nn import CTCLoss

ctc_loss = CTCLoss(reduction='none')

batch_size = ctc_logits.shape[0]

# replace with the actual encoded keywords from the batch computed above
encoded_keyword = tokenizer.encode('kúkùŋú')
targets = torch.tensor([encoded_keyword]*batch_size)
target_lengths = torch.tensor([len(encoded_keyword)]*batch_size)

# TODO: cast logits into probabilities
ctc_probs = ctc_logits # your code here

loss = ctc_loss(
    # permute from batch_size, length, vocab_size -> length, batch_size, vocab_size
    # to match the input shape expected by CTC loss
    ctc_probs.permute(1, 0, 2),
    targets,
    output_lengths,
    target_lengths,
)
loss, loss.shape

(tensor([313.6105, 316.3583, 317.9232, 314.7330, 275.2854, 295.9673, 296.8927,
         323.7849, 313.7808, 309.4163, 308.2051, 305.3466, 288.7700, 305.2484,
         305.4482, 258.8286, 279.2108, 282.6724, 273.5536, 264.7640, 253.0972,
         254.3431, 262.6536, 248.7715, 235.9852, 234.5585, 231.4681, 200.2018,
         190.8554, 178.5805, 184.2073, 165.9626], device='cuda:0'),
 torch.Size([32]))

## Label probability vs keyword probability
So now we've got the loss, which is equivalent to the probability of the label given the audio.
But this isn't enough for KWS.
We're not computing the probability that the audio *is* the keyword but that the audio *contains* the keyword.
To do this, we change the label so that any speech **OR** silence can be matched before or after the keyword.
This is the keyword/background model from last Fall.

How does CTC loss relate to keyword probability, and what is missing to convert it to KWS probability?
Hint: CTC loss is calculated by the intersection of two WFST graphs.
What are those graphs and what do they represent?
When doing KWS, how do the graphs differ?
I'd suggest drawing the graphs, and their resulting intersection, out using pen and paper.

Refer to [Xi et al 2025](10.1109/ICASSP49660.2025.10889332) for more explanation of KWS graph decoding.